# Evaluating Test Notebook
This notebook verifies the evaluation stage, ensuring all metrics (Runtime, Deletion Ratio, Marginals Error, TVD, and ML Accuracy) are calculated and saved correctly.

## 1. Setup & Mini Pipeline

In [ ]:
import sys
import os
import time
sys.path.append(os.path.abspath('..'))

from src.loading.file_loader import FileLoader
from src.loading.components.data_loader import DataLoader
from src.loading.components.dcs_loader import DCsLoader
from src.loading.components.metadata_loader import MetadataLoader
from src.loading.components.data_encoder import DataEncoder
from src.loading.components.dcs_encoder import DCsEncoder
from src.synthesizing.co_noise import CoNoise
from src.marginals_obtaining.top_k_obtainer import TopKObtainer
from src.marginals_obtaining.utility_functions.distance_utility import DistanceUtility
from src.repairing.ilp_repairer import ILPRepairer
from src.entities.pipeline_result import PipelineResult
from src.evaluating.evaluation_orchestrator import EvaluationOrchestrator
from src.evaluating.runtime_evaluator import RuntimeEvaluator
from src.evaluating.deletion_ratio_evaluator import DeletionRatioEvaluator
from src.evaluating.marginals_error_evaluator import MarginalsErrorEvaluator
from src.evaluating.tvd_evaluator import TwoWayTVDEvaluator
from src.evaluating.ml_accuracy_evaluator import MLAccuracyEvaluator

def run_mini_pipeline(dataset_name):
    runtimes = {}
    
    # 1. Loading
    start = time.time()
    loader = FileLoader(
        name=dataset_name, base_path="../data",
        data_loader=DataLoader(), dcs_loader=DCsLoader(), metadata_loader=MetadataLoader(),
        data_encoder=DataEncoder(), dcs_encoder=DCsEncoder()
    )
    private_ds = loader.load()
    runtimes['loading'] = time.time() - start
    
    # 2. Synthesizing
    start = time.time()
    synthesizer = CoNoise(num_of_iterations=50)
    synthetic_ds = synthesizer.synthesize(private_ds)
    runtimes['synthesizing'] = time.time() - start
    
    # 3. Marginals
    start = time.time()
    obtainer = TopKObtainer(selection_budget=1.0, generation_budget=1.0, k=5, utility_function=DistanceUtility())
    marginals = obtainer.obtain(private_ds, synthetic_ds)
    runtimes['marginals'] = time.time() - start
    
    # 4. Repairing
    start = time.time()
    repairer = ILPRepairer(alpha=0.8, use_marginals=True, gurobi_params={"OutputFlag": 0, "TimeLimit": 30})
    repaired_ds = repairer.repair(synthetic_ds, marginals)
    runtimes['repairing'] = time.time() - start
    
    return PipelineResult(
        private_dataset=private_ds,
        synthetic_dataset=synthetic_ds,
        repaired_dataset=repaired_ds,
        obtained_marginals=marginals,
        runtimes=runtimes
    )

result = run_mini_pipeline("tax")

## 2. Run Evaluation

In [ ]:
evaluators = [
    RuntimeEvaluator(),
    DeletionRatioEvaluator(),
    MarginalsErrorEvaluator(),
    TwoWayTVDEvaluator(),
    MLAccuracyEvaluator()
]

orchestrator = EvaluationOrchestrator(evaluators, output_dir="test_results")
eval_results = orchestrator.run(result)

import pprint
pprint.pprint(eval_results)

## 3. Verify Output File

In [ ]:
files = os.listdir("test_results")
print(f"Files in test_results: {files}")
assert len(files) > 0, "Result file was not created!"

with open(os.path.join("test_results", files[0]), "r") as f:
    content = json.load(f)
    assert "deletion_ratio" in content
    assert "tvd_2way" in content
    assert "ml_accuracy" in content
    print("Evaluation content verified!")